# Phase 4 — Distribution Shift Detection

**Goal:** Identify which test compounds the model is least confident about, and analyze whether confidence differs across splits A (random), B (scaffold shift), and C (assay shift).

This analysis is critical for the **methodology report** (50% of your competition grade). We show that our pipeline can detect when it's out of its comfort zone.

### Two signals we combine:
1. **Mahalanobis distance** in GNN embedding space — how structurally different is this compound from training compounds?
2. **MC Dropout epistemic uncertainty** — how much does the model disagree with itself across stochastic passes?

In [14]:
!pwd

/Users/shaqbeast/STAR-AI-Competition/DTA-Multiagent-Pipeline


## 1. Setup

In [5]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_mean_pool
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.covariance import EmpiricalCovariance
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [6]:
# Shared config across Phase 3/4/5
INPUT_DIR    = 'data/'
MODEL_DIR    = 'saved_models_v2/'
ENSEMBLE_SIZE = 5
HIDDEN_DIM   = 256
ESM_DIM      = 1280
NODE_FEAT    = 29
EDGE_FEAT    = 7
N_GIN_LAYERS = 4
DROPOUT      = 0.2
BATCH_SIZE   = 128

## 2. Load Model & Data

In [9]:
class DTA_Model(nn.Module):
    def __init__(self, node_feat=29, edge_feat=7, esm_dim=1280,
                 hidden_dim=256, n_layers=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        # --- Drug graph encoder (GINEConv) ---
        # NO manual edge_proj — GINEConv handles it via edge_dim
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for i in range(n_layers):
            in_dim = node_feat if i == 0 else hidden_dim
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            # edge_dim tells GINEConv to internally project edges to match nodes
            self.convs.append(GINEConv(mlp, edge_dim=edge_feat))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        # --- rest is the same ---
        self.fp_proj = nn.Sequential(
            nn.Linear(2048, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
        )
        self.prot_proj = nn.Sequential(
            nn.Linear(esm_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def get_compound_embedding(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_attr)  # pass raw edge_attr directly
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return global_mean_pool(x, batch)

    def forward(self, data):
        drug_graph = self.get_compound_embedding(data)
        drug_fp = self.fp_proj(data.fp.squeeze(1))
        prot = self.prot_proj(data.target_emb.squeeze(1))
        combined = torch.cat([drug_graph, drug_fp, prot], dim=-1)
        return self.head(combined).squeeze(-1)

In [11]:
# Load first ensemble member (all share the same architecture)
model = DTA_Model(node_feat=NODE_FEAT, edge_feat=EDGE_FEAT, esm_dim=ESM_DIM,
                  hidden_dim=HIDDEN_DIM, n_layers=N_GIN_LAYERS, dropout=DROPOUT).to(device)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'model_1.pt'), map_location=device))
print('Model loaded.')

train_data = torch.load(os.path.join(INPUT_DIR, 'train_data.pt'))
val_data   = torch.load(os.path.join(INPUT_DIR, 'val_data.pt'))
test_data  = torch.load(os.path.join(INPUT_DIR, 'test_data.pt'))

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')

Model loaded.
Train: 23194 | Val: 2578 | Test: 23191


## 3. Compute Murcko Scaffolds

Phase 2 didn't compute scaffolds, so we derive them from the SMILES in the unified CSV. The scaffold is the core ring skeleton — like the floor plan of a house, ignoring the paint and landscaping.

In [ ]:
df_unified = pd.read_csv("unified_dta_scaffold_split.csv")

# Build scaffold lookup using the pre-computed Scaffold column.
# train sample_ids are "train_XXXXX" where XXXXX is the row index within the Train split.
# Test sample_ids are competition IDs (D_XXXXX) not in the CSV, so test splits come
# from the in_A/in_B/in_C flags stored on each data object.
train_df = df_unified[df_unified["Split"] == "Train"].reset_index(drop=True)

def get_train_scaffold(sample_id):
    try:
        idx = int(sample_id.split("_", 1)[1])
        return train_df.iloc[idx]["Scaffold"]
    except Exception:
        return "unknown"

print("Building scaffold lookup from CSV...")
train_scaffolds = [get_train_scaffold(d.sample_id) for d in train_data]

# Test SMILES are not in the CSV, so scaffold analysis is unavailable for test.
test_scaffolds = ["unknown"] * len(test_data)

# Derive test sub-split labels from the in_A/in_B/in_C flags on each data object.
test_splits = []
for d in test_data:
    if d.in_A.item() == 1:
        test_splits.append("A")
    elif d.in_B.item() == 1:
        test_splits.append("B")
    elif d.in_C.item() == 1:
        test_splits.append("C")
    else:
        test_splits.append("unknown")

known_scaffolds = set(s for s in train_scaffolds if s not in ("unknown", "invalid"))
print(f"Unique train scaffolds: {len(known_scaffolds)}")
print(f'Test samples: {len(test_data)} | A={test_splits.count("A")}, B={test_splits.count("B")}, C={test_splits.count("C")}')


FileNotFoundError: [Errno 2] No such file or directory: 'unified_dta_competition_split.csv'

## 4. Extract GNN Compound Embeddings

We use the `get_compound_embedding()` method to pull out the 256-d vectors the model uses internally. Structurally similar molecules cluster together in this space.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, device):
    model.eval()
    embeddings = []
    for batch in loader:
        batch = batch.to(device)
        h = model.get_compound_embedding(batch)
        embeddings.append(h.cpu().numpy())
    return np.concatenate(embeddings)

print('Extracting embeddings...')
train_embeds = extract_embeddings(model, train_loader, device)
test_embeds  = extract_embeddings(model, test_loader, device)
print(f'Train: {train_embeds.shape} | Test: {test_embeds.shape}')

## 5. Mahalanobis Distance — Structural Novelty Score

We cluster training embeddings, fit a covariance per cluster, then measure how far each test compound is from its nearest cluster. Far = structurally novel = less trustworthy prediction.

In [ ]:
N_CLUSTERS = 10
print(f'Clustering training embeddings into {N_CLUSTERS} clusters...')
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
train_labels = kmeans.fit_predict(train_embeds)
centroids = kmeans.cluster_centers_

precision_matrices = []
for k in range(N_CLUSTERS):
    mask = train_labels == k
    embeds = train_embeds[mask]
    if len(embeds) < train_embeds.shape[1] + 1:
        embeds = train_embeds  # fallback to global
    cov = EmpiricalCovariance().fit(embeds)
    precision_matrices.append(cov.precision_)

def batch_mahalanobis(embeddings, centroids, precisions):
    dists = []
    for emb in embeddings:
        d_min = min(
            np.sqrt(np.clip((emb - c) @ P @ (emb - c), 0, None))
            for c, P in zip(centroids, precisions)
        )
        dists.append(d_min)
    return np.array(dists)

print('Computing Mahalanobis distances...')
test_mahal = batch_mahalanobis(test_embeds, centroids, precision_matrices)
print(f'Test distances: mean={test_mahal.mean():.2f}, std={test_mahal.std():.2f}')

## 6. MC Dropout — Epistemic Uncertainty

Run the model 30 times with dropout active. The variance across passes tells us how uncertain the model is about each compound.

In [ ]:
def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

N_MC = 30
model.eval()
enable_mc_dropout(model)

print(f'Running {N_MC} MC Dropout passes on test set...')
all_passes = []
for t in range(N_MC):
    preds = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            preds.extend(model(batch).cpu().numpy())
    all_passes.append(preds)

all_passes = np.array(all_passes)  # (30, N_test)
test_epistemic = all_passes.var(axis=0)
print(f'Epistemic uncertainty: mean={test_epistemic.mean():.4f}, max={test_epistemic.max():.4f}')

## 7. Combined OOD Score & Per-Split Analysis

Combine both signals, then break down by split. We expect Split B (scaffold shift) to show higher Mahalanobis distances and Split C (assay shift) to show higher model uncertainty.

In [ ]:
# Normalize and combine
def normalize_01(arr):
    p5, p95 = np.percentile(arr, 5), np.percentile(arr, 95)
    return np.clip((arr - p5) / (p95 - p5 + 1e-8), 0, 1)

mahal_norm = normalize_01(test_mahal)
ep_norm = normalize_01(test_epistemic)
ood_score = 0.5 * mahal_norm + 0.5 * ep_norm

# Build results (novel_scaffold omitted: test SMILES not available in the CSV)
test_ids = [d.sample_id for d in test_data]
results_df = pd.DataFrame({
    'sample_id': test_ids,
    'split': test_splits,
    'mahalanobis': test_mahal,
    'epistemic': test_epistemic,
    'ood_score': ood_score,
})

# Per-split summary
print('=== Per-Split OOD Analysis ===')
for s in ['A', 'B', 'C']:
    sub = results_df[results_df['split'] == s]
    if len(sub) == 0: continue
    m = sub['mahalanobis'].mean()
    e = sub['epistemic'].mean()
    o = sub['ood_score'].mean()
    print(f'\nSplit {s} ({len(sub)} samples):')
    print(f'  Mahalanobis:  mean={m:.2f}')
    print(f'  Epistemic:    mean={e:.4f}')
    print(f'  OOD score:    mean={o:.3f}')


## 8. Visualization — For Your Report

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Embedding space PCA
all_emb = np.vstack([train_embeds, test_embeds])
pca = PCA(n_components=2)
all_2d = pca.fit_transform(all_emb)
train_2d = all_2d[:len(train_embeds)]
test_2d = all_2d[len(train_embeds):]

axes[0].scatter(train_2d[:, 0], train_2d[:, 1], c='lightgray', s=2, alpha=0.2, label='Train')
split_arr = np.array(test_splits)
for s, color, marker in [('A', 'tab:blue', 'o'), ('B', 'tab:orange', 's'), ('C', 'tab:green', '^')]:
    mask = split_arr == s
    if mask.sum() > 0:
        axes[0].scatter(test_2d[mask, 0], test_2d[mask, 1], c=color, s=8, alpha=0.4, label=f'Test {s}', marker=marker)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[0].set_title('Embedding Space by Split')
axes[0].legend(fontsize=8)

# 2. OOD score by split
split_data = []
split_labels = []
for s in ['A', 'B', 'C']:
    mask = split_arr == s
    if mask.sum() > 0:
        split_data.append(ood_score[mask])
        split_labels.append(f'Split {s}')
axes[1].boxplot(split_data, labels=split_labels)
axes[1].set_ylabel('OOD Score')
axes[1].set_title('OOD Score Distribution by Split')

# 3. Mahalanobis vs epistemic
colors = {'A': 'tab:blue', 'B': 'tab:orange', 'C': 'tab:green'}
for s in ['A', 'B', 'C']:
    mask = split_arr == s
    if mask.sum() > 0:
        axes[2].scatter(test_mahal[mask], test_epistemic[mask], c=colors[s],
                        alpha=0.3, s=8, label=f'Split {s}')
axes[2].set_xlabel('Mahalanobis Distance (structural novelty)')
axes[2].set_ylabel('Epistemic Uncertainty (model disagreement)')
axes[2].set_title('Two OOD Signals by Split')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('phase4_ood_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Results

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
results_df.to_csv(os.path.join(MODEL_DIR, 'phase4_ood_results.csv'), index=False)

torch.save({
    'centroids': centroids,
    'precision_matrices': precision_matrices,
    'known_scaffolds': list(known_scaffolds),
}, os.path.join(MODEL_DIR, 'phase4_detector_state.pt'))

print('Saved:')
print(f'  {MODEL_DIR}phase4_ood_results.csv')
print(f'  {MODEL_DIR}phase4_detector_state.pt')
print('\nPhase 4 complete → proceed to Phase 5.')